In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/supplement-extra"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import decoupler as dc
import matplotlib.pyplot as plt
import scanpy as sc

from spatial_tcr.tcr import get_tcr_genes

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad"
adata = sc.read_h5ad(path)
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

In [ ]:
cc_key = "avbv_cluster_filtered"
infiltrate_key = "tcell_infiltrate"

In [ ]:
adata.obs[infiltrate_key].value_counts()

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()
ad_t.obs["in_clonal_cluster"] = (
    ad_t.obs[cc_key].isin(ad_t.obs[cc_key].dropna().unique()).astype(str)
)

# remove trv genes
tv_genes = get_tcr_genes(ad_t)[-1]
ad_t = ad_t[:, [g for g in ad_t.var_names if g not in tv_genes]].copy()

ad_t.X = ad_t.layers["counts"].copy()
sc.pp.normalize_total(ad_t)
sc.pp.log1p(ad_t)

In [ ]:
ad_t_infilt = ad_t[ad_t.obs[infiltrate_key] == "infiltrate"].copy()
ad_t_non_infilt = ad_t[ad_t.obs[infiltrate_key] == "no infiltrate"].copy()

In [ ]:
figsize = (4.5, 3.5)

In [ ]:
sc.tl.rank_genes_groups(
    ad_t_infilt, groupby="in_clonal_cluster", method="wilcoxon", use_raw=False
)
# sc.pl.rank_genes_groups_dotplot(
#     ad_t_infilt, groupby="in_clonal_cluster", standard_scale="var", n_genes=10
# )
de_df = sc.get.rank_genes_groups_df(ad_t_infilt, group="True").set_index("names")
fig, ax = plt.subplots(1, 1, figsize=figsize)
dc.pl.volcano(de_df, x="logfoldchanges", y="pvals_adj", ax=ax, top=15, thr_stat=0.4)
ax.set_xlim(-5, 5)
ax.tick_params(axis="both", labelsize=8)
ax.xaxis.label.set_size(10)
ax.yaxis.label.set_size(10)
# ax.set_xlim(-5, 5)
ax.set_title(
    "DEGs between T cells in clonal clusters\nand other T cells (inside T cell infiltrate)",
    fontsize=10,
)
ax.set_ylabel("-log10(p-adjusted)", fontsize=10)
# fig.subplots_adjust(left=0.2)
# fig.tight_layout()
plt.savefig(
    f"{figure_dir}/DEGs_clonal_cluster_inside_tcell_infiltrate.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
de_df = sc.get.rank_genes_groups_df(ad_t_infilt, group="True").set_index("names")
log2fc_cutoff = 0.0
pvalue_cutoff = 0.05
mask_1 = de_df["pvals_adj"] < pvalue_cutoff
mask_2 = de_df["logfoldchanges"] > log2fc_cutoff
# mask_3 = de_df["logfoldchanges"] < -log2fc_cutoff
mask = mask_1 & mask_2
de_df = de_df.loc[mask]
de_df

In [ ]:
# differentially expressed genes between expanded and other T cells
sc.tl.rank_genes_groups(
    ad_t_non_infilt, groupby="in_clonal_cluster", method="wilcoxon", use_raw=False
)
de_df = sc.get.rank_genes_groups_df(ad_t_non_infilt, group="True").set_index("names")
fig, ax = plt.subplots(1, 1, figsize=figsize)
dc.pl.volcano(de_df, x="logfoldchanges", y="pvals_adj", ax=ax, top=15, thr_stat=0.4)
ax.set_xlim(-5, 5)
ax.tick_params(axis="both", labelsize=8)
ax.xaxis.label.set_size(10)
ax.yaxis.label.set_size(10)
ax.set_title(
    "DEGs between T cells in clonal clusters\nand other T cells (outside T cell infiltrate)"
)
ax.set_ylabel("-log10(p-adjusted)", fontsize=10)
plt.savefig(
    f"{figure_dir}/DEGs_clonal_cluster_outside_tcell_infiltrate.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
de_df = sc.get.rank_genes_groups_df(ad_t_non_infilt, group="True").set_index("names")
log2fc_cutoff = 0.0
pvalue_cutoff = 0.05
mask_1 = de_df["pvals_adj"] < pvalue_cutoff
mask_2 = de_df["logfoldchanges"] > log2fc_cutoff
# mask_3 = de_df["logfoldchanges"] < -log2fc_cutoff
mask = mask_1 & mask_2
de_df = de_df.loc[mask]
de_df

## Also compare inside and outside T cell infiltrate

In [ ]:
ad_cluster = ad_t[ad_t.obs["in_clonal_cluster"] == "True"].copy()
ad_other_t = ad_t[ad_t.obs["in_clonal_cluster"] == "False"].copy()

In [ ]:
ad_t.obs[infiltrate_key].value_counts()

In [ ]:
sc.tl.rank_genes_groups(
    ad_cluster, groupby=infiltrate_key, method="wilcoxon", use_raw=False
)

de_df = sc.get.rank_genes_groups_df(ad_cluster, group="infiltrate").set_index("names")
fig, ax = plt.subplots(1, 1, figsize=figsize)
dc.pl.volcano(de_df, x="logfoldchanges", y="pvals_adj", ax=ax, top=15, thr_stat=0.4)
ax.set_xlim(-5, 5)
ax.tick_params(axis="both", labelsize=8)
ax.xaxis.label.set_size(10)
ax.yaxis.label.set_size(10)
# ax.set_xlim(-5, 5)
ax.set_title("DEGs between clonal cluster T cells\nin- and outside T cell infiltrates")
ax.set_ylabel("-log10(p-adjusted)", fontsize=10)
plt.savefig(
    f"{figure_dir}/DEGs_clonal_cluster_inside_outside_tcell_infiltrate.pdf",
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
sc.tl.rank_genes_groups(
    ad_other_t, groupby=infiltrate_key, method="wilcoxon", use_raw=False
)

de_df = sc.get.rank_genes_groups_df(ad_other_t, group="infiltrate").set_index("names")
fig, ax = plt.subplots(1, 1, figsize=figsize)
dc.pl.volcano(de_df, x="logfoldchanges", y="pvals_adj", ax=ax, top=15, thr_stat=0.4)
ax.set_xlim(-5, 5)
ax.tick_params(axis="both", labelsize=8)
ax.xaxis.label.set_size(10)
ax.yaxis.label.set_size(10)
# ax.set_xlim(-5, 5)
ax.set_title("DEGs between unclustered T cells\nin- and outside T cell infiltrates")
ax.set_ylabel("-log10(p-adjusted)", fontsize=10)
plt.savefig(
    f"{figure_dir}/DEGs_tcell_inside_outside_tcell_infiltrate.pdf",
    dpi=300,
    bbox_inches="tight",
)